In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# --- SCHRITT 1: Daten laden ---
df = pd.read_csv("Datasets/skoda.csv")  # Pfad ggf. anpassen


In [ ]:
# --- SCHRITT 1.1: EDA (Explorative Datenanalyse) ---
print("=== Erste Zeilen ===")
print(df.head(5))
print()

print("=== Beschreibende Statistik ===")
print(df.describe())
print()

print("=== Fehlende Werte ===")
print(df.isnull().sum())
print()

# Verteilung der Zielvariable
plt.figure(figsize=(8, 4))
plt.hist(df["price"], bins=50, edgecolor="black", color="steelblue")
plt.xlabel("Preis (£)")
plt.ylabel("Häufigkeit")
plt.title("Verteilung des Preises")
plt.tight_layout()
plt.show()

# Korrelationsmatrix (nur numerische Spalten der Rohdaten)
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Korrelationsmatrix (Rohdaten)")
plt.tight_layout()
plt.show()

In [ ]:
# --- SCHRITT 2: Preprocessing ---
# --- 2a: Kategorische Spalten – vorkommende Werte anzeigen ---
KATEGORISCH = ["model", "transmission", "fuelType"]

print("=== Eindeutige Werte je kategorischer Spalte ===")
for col in KATEGORISCH:
    werte = sorted(df[col].unique())
    print(f"  {col} ({len(werte)} Werte): {werte}")

In [ ]:
# --- 2b: Filterwerte hard coden & Datensatz filtern ---
# Jede Spalte wird auf genau einen Wert gefiltert → Spalte wird danach gedroppt.

df_filtered = df[
    (df["model"]        == ' Octavia') &    # WERT ANPASSEN
    (df["transmission"] == 'Manual') &      # WERT ANPASSEN
    (df["fuelType"]     == 'Petrol')        # WERT ANPASSEN
].copy()

# Da jede Spalte auf genau einen Wert gefiltert wurde, sind alle konstant → droppen.
df_filtered = df_filtered.drop(columns=KATEGORISCH)

print(f"Zeilen vor Filter:  {len(df)}")
print(f"Zeilen nach Filter: {len(df_filtered)}")

In [ ]:
# --- 2c: Feature Engineering ---

# 'year' → 'alter': Datensatz stammt aus dem Jahr 2020
DATENSATZ_JAHR = 2020
df_filtered["alter"] = DATENSATZ_JAHR - df_filtered["year"]

# 'mpg' → 'verbrauch' (Liter / 100 km)
# Umrechnung britische Gallonen: L/100km = 282.481 / mpg
df_filtered["verbrauch"] = 282.481 / df_filtered["mpg"]

# Originalspalten entfernen
df_filtered = df_filtered.drop(columns=["year", "mpg"])

print(df_filtered[["alter", "verbrauch"]].describe())

In [ ]:
# --- 2d: Ausreißer prüfen (Boxplots der numerischen Spalten) ---
numerisch = [col for col in df_filtered.select_dtypes(include=np.number).columns
             if col != "price"]

fig, axes = plt.subplots(1, len(numerisch), figsize=(16, 4))
for ax, col in zip(axes, numerisch):
    ax.boxplot(df_filtered[col].dropna())
    ax.set_title(col)
plt.suptitle("Boxplots – Numerische Features")
plt.tight_layout()
plt.show()

# Ausreißer in 'price' via IQR entfernen
Q1 = df_filtered["price"].quantile(0.25)
Q3 = df_filtered["price"].quantile(0.75)
IQR = Q3 - Q1
df_clean = df_filtered[
    (df_filtered["price"] >= Q1 - 1.5 * IQR) &
    (df_filtered["price"] <= Q3 + 1.5 * IQR)
]

print(f"Datensätze vor  Ausreißer-Entfernung: {len(df_filtered)}")
print(f"Datensätze nach Ausreißer-Entfernung: {len(df_clean)}")

In [ ]:
# --- 2e: X / y trennen ---
X = df_clean.drop(columns=["price"])
y = df_clean["price"]

print(f"Features ({X.shape[1]}): {list(X.columns)}")
print(f"Zielvariable: price  ({len(y)} Beobachtungen)")

In [ ]:
# --- SCHRITT 3: Der "Goldene Schnitt" (Train-Test-Split) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Trainingsdaten:  {X_train.shape[0]} Zeilen")
print(f"Testdaten:       {X_test.shape[0]} Zeilen")

In [ ]:
# --- SCHRITT 4: Die Pipeline bauen ---
# Random Forest ist baumbasiert → kein StandardScaler nötig.

pipeline = Pipeline([
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

print(pipeline)

In [ ]:
# --- SCHRITT 5: K-Fold Cross Validation (Training & Validierung) ---
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores_r2   = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="r2")
scores_rmse = cross_val_score(pipeline, X_train, y_train, cv=cv,
                               scoring="neg_root_mean_squared_error")

print("--- Cross-Validation Ergebnisse (5 Folds) ---")
print(f"R² je Fold:           {np.round(scores_r2, 4)}")
print(f"Ø R²:                 {np.mean(scores_r2):.4f}  (+/- {np.std(scores_r2):.4f})")
print()
print(f"RMSE je Fold (£):     {np.round(-scores_rmse, 0)}")
print(f"Ø RMSE (£):           {-np.mean(scores_rmse):.0f}  (+/- {np.std(scores_rmse):.0f})")

In [ ]:
# --- SCHRITT 6: Das finale Modell trainieren ---
pipeline.fit(X_train, y_train)

In [ ]:
# --- SCHRITT 7: Finale Evaluation (Die Stunde der Wahrheit) ---
y_pred = pipeline.predict(X_test)

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("--- Finale Evaluation auf dem Test-Set ---")
print(f"R²:    {r2:.4f}   (1.0 = perfekt)")
print(f"MAE:   £{mae:,.0f}   (Ø absoluter Fehler)")
print(f"RMSE:  £{rmse:,.0f}  (Ø quadratischer Fehler)")

# Residuenplot
residuen = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred, residuen, alpha=0.3, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Vorhergesagter Preis (£)")
axes[0].set_ylabel("Residuum (£)")
axes[0].set_title("Residuenplot")

axes[1].scatter(y_test, y_pred, alpha=0.3, color="steelblue")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[1].plot(lims, lims, "r--", label="Ideal")
axes[1].set_xlabel("Tatsächlicher Preis (£)")
axes[1].set_ylabel("Vorhergesagter Preis (£)")
axes[1].set_title("Vorhersage vs. Realität")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- SCHRITT 7.1: Feature Importance ---
importances = pipeline.named_steps["regressor"].feature_importances_
feat_df = pd.DataFrame({"Feature": X.columns, "Importance": importances})
feat_df = feat_df.sort_values("Importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feat_df["Feature"], feat_df["Importance"], color="steelblue")
plt.xlabel("Importance")
plt.title("Feature Importance – Random Forest")
plt.tight_layout()
plt.show()

print(feat_df.to_string(index=False))

In [ ]:
# --- SCHRITT 8: Modell speichern (Deployment) ---
joblib.dump(pipeline, "Modelle/skoda_preismodell_rf.pkl")
print("Modell wurde als 'skoda_preismodell_rf.pkl' gespeichert.")